In [1]:
from IPython.display import Image

$$
\text{最终输出分布}=\text{大模型原本的输出分布}
$$
> 证明任意一个 Token $x$ 被系统最终采样的总概率 $P(x)$ 严格等于目标分布 $p(x)$（不只是期望意义上，或者叫无损的）。最终输出 token 的完整分布与目标模型分布 p 完全一致。modified rejection sampling preserves the distribution of the target model。
- notations
    - 假设 $p(x)$ 为庞大且缓慢的目标模型（Target Model）对下一个 Token 的概率分布。
    - 假设 $q(x)$ 为轻量且快速的草稿模型（Draft Model）对下一个 Token 的概率分布。
- 解码流程
    - 提议（Propose）：从草稿模型中采样一个候选 Token $x \sim q(x)$。
    - 验证与接受（Verify & Accept）：目标模型并行计算其概率 $p(x)$，并以概率 $\alpha$ 接受该候选，其中接受率定义为：
        - $\alpha(x) = \min\left(1, \frac{p(x)}{q(x)}\right)$
        - token $x$ 被“草稿接受”的概率质量是：$q(x)a(x)=q(x)\min\left(1,\frac{p(x)}{q(x)}\right)=\min(q(x),p(x))$
            - 草稿模型贡献的那部分概率质量，恰好是 $p$ 与 $q$ 的重叠部分。
    - 拒绝与重采样（Reject & Resample）：如果 $x$ 被拒绝，系统将直接丢弃 $q(x)$ 的当前建议，并从一个经过修正的残差分布（Residual Distribution） $p'(x)$ 中重新采样最终的 Token。该残差分布定义为目标分布扣除被接受部分后的归一化结果：
        - $p'(x) = \frac{\max(0, p(x) - q(x))}{\sum_{x'} \max(0, p(x') - q(x'))}$
- 全概率公式的视角: $p(x)$
    - 接受: $q(x)\cdot\min(1,\frac{p(x)}{q(x)})=\min(q(x), p(x))$
        - 直接接受了 draft model 的 $q(x)$
    - 拒绝: $p(x)-\min(p(x), q(x))$
        - $q(\cdot)$ 采样了 $x'$ 被拒绝，依重采样后的概率采样到了 $x$
    - 总：$\min(q(x),p(x)) + (p(x) - \min(p(x), q(x)))=p(x)$

### 探测解码分布一致性推导

> 证明任意一个 Token $x$ 被系统最终采样的总概率 $P(x)$ 严格等于目标分布 $p(x)$。
- 路径 A（草稿命中并被接受）：$x$ 最初被草稿模型抽中，并且顺利通过了目标模型的接受测试。
    - $P(A) = q(x) \cdot \min\left(1, \frac{p(x)}{q(x)}\right) = \min(q(x), p(x))$
- 路径 B（拒绝后从残差中补偿）：草稿模型抽中了某个候选（可能是任意的 $x'$）但遭到了拒绝；随后在修正分布 $p'(x)$ 中，恰好抽中了 $x$。
    - $P(\text{reject}) = \sum_{x'} q(x') \left(1 - \min\left(1, \frac{p(x')}{q(x')}\right)\right) = \sum_{x'} \max(0, q(x') - p(x')) = \sum_{x'} \max(0, p(x') - q(x'))$
        - $a - \min(a, b) = \max(0, a - b)$ （$\max(0, q(x') - p(x'))$，草稿模型，自作多情多给出的概率质量）
        - $a - b = \max(0, a-b) - \max(0, b-a)$
        - $q(x') - p(x') = \max(0, q(x') - p(x')) - \max(0, p(x') - q(x'))$
    - 从残差分布中重采样抽中 $x$ 的联合概率为：
        - $P(B) = P(\text{reject}) \cdot p'(x) = P(\text{reject}) \cdot \frac{\max(0, p(x) - q(x))}{P(\text{reject})} = \max(0, p(x) - q(x))$
- 全概率公式：$P(x) = P(A) + P(B) = \min(q(x), p(x)) + \max(0, p(x) - q(x))=p(x)$
    - $\min(a,b) + \max(0, a-b) \equiv a$

- 如果小模型高估了某个词的概率（$q > p$）：过度自信
    - 小模型太爱出这个词了。为了纠正它，大模型在验证时，会按比例 $\frac{p}{q}$砍掉多余的部分（拒绝掉）。剩下的概率刚好是 $p$。而且在重采样时，因为 $p-q\lt 0$，大模型绝不会再主动生成这个词。（被拒绝的不可能再被接受）
- 如果小模型低估了某个词的概率（$q\lt p$） ：自信不足
    - 小模型不太出这个词，但只要它出了，大模型觉得“这词很好”，会 $100\%$ 接受($\alpha=1$)。然而这还不够，大模型缺少的概率份额 $p-q$，会在大模型拒绝其他词后，通过重采样（Resampling）环节补齐。

#### examples

> 被拒绝的不可能再被接受
- 假设词表只有三个 token（$x_1,x_2,x_3$）：
    - $p=(0.5,0.3,0.2)$
    - $q=(0.6,0.1,0.3)$
- 草稿接受部分是
    - $q\cdot\min(1,\frac{p}{q})=\min(p,q)=(0.5,0.1,0.2)$
    - 接受总概率是: $0.5+0.1+0.2=0.8$
    - 拒绝概率是: $0.2$ ($P(\text{reject})$)
- 补偿分布来自 $p−q$ 的正部：$(p-q)_+=(0,0.2,0)$
    - 归一化后，$r=\frac{(p-q)_+}{\sum_i (p_i-q_i)_+}=\frac{(0,0.2,0)}{0.2}=(0,1,0)$ ($p'(x)$)
    - 一旦拒绝，就一定输出 $x_2$;一旦拒绝，就一定补采样到 $x_2$。
- 所以拒绝时必定补回第二个 token。最终分布是
    - $(0.5,0.1,0.2)+0.2(0,1,0)=(0.5,0.3,0.2)=p$
-----
全概率公式视角，设草稿采样结果是 $D$，最终输出是 $Y$。对任意 token $x_i$：

$$
\begin{split}
\Pr(Y=x_i)&=\Pr(D=x_i,\ \text{接受})+\sum_j \Pr(D=x_j,\ \text{拒绝},\ \text{重采样到 }x_i)\\
&=q_i a_i+\left(\sum_j q_j(1-a_j)\right)p_{\text{res}}(x_i)
\end{split}
$$
- $q_j(1-a_j)=q_j-q_ja_j=q_j-\min(q_j,p_j)$
- $\Pr(Y=x_2)=q_2a_2+q_1(1-a_1)p_{\text{res}}(x_2)+q_2(1-a_2)p_{\text{res}}(x_2)+q_3(1-a_3)p_{\text{res}}(x_2)$
    - $\Pr(Y=x_2)=0.1+0.1\cdot 1+0\cdot 1+0.1\cdot 1=0.3$
- $q$ 对 $x_1,x_3$ 给多了，所以接受-拒绝（accept-reject）步骤削掉多余质量（PMF）；$q$ 对 $x_2$ 给少了，所以拒绝后的残差采样把少的那部分 $0.2$ 精确补给 $x_2$。这就是为什么最终分布仍然严格等于目标分布 $p$。

- 词表有4个tokens
    - $p=(0.4,0.1,0.3,0.2)$
    - $q=(0.2,0.3,0.1,0.4)$
    - $q$ 在 $t_2,t_4$ 上给多了，在 $t_1,t_3$上给少了。投机采样会拒绝一部分 $t_2,t_4$，然后把这些概率质量补给 $t_1,t_3$
- 接受概率 $\alpha=(1,\frac13,1,\frac12)$，拒绝概率 $(0,\frac23,0,\frac12)$
    - 采到 $t_1,t_3$时一定接受；采到 $t_2,t_4$ 时可能拒绝；
- 接受路径的概率质量：$q\alpha=(0.2, 0.1, 0.1, 0.2)$
- $(p-q)_+=(0.2,0,0.2,0)$
- 归一化 $r=\frac{(0.2,0,0.2,0)}{0.2+0.2}=(0.5,0,0.5,0)$
    - 所以一旦进入拒绝重采样，最终只可能补采到 $t_1$ 和 $t_3$，各占一半。
- 最终分布是
    - $(0.2, 0.1, 0.1, 0.2) + 0.4(0.5,0,0.5,0)=(0.4,0.1,0.3,0.2)$

### MTP

- mimo v2 flash paper
- hf
    - mimo v2 flash: https://huggingface.co/XiaomiMiMo/MiMo-V2.5
    - qwen3.6-27b: https://huggingface.co/Qwen/Qwen3.6-27B

In [4]:
# https://sebastianraschka.com/llm-architecture-gallery/mtp/
Image(url='https://sebastianraschka.com/llm-architecture-gallery/images/concepts/mtp-next-token-vs-multi-token.webp',
      width=600)

> teacher forcing

- 训练数据：$x_1,x_2,\ldots,x_T$ 
- 标准 next-token prediction 只优化：$p(x_{t+1}\mid x_{\le t})$
    - $L_{\text{next}}=-\frac{1}{T-1}\sum_{t=1}^{T-1}\log p_{\theta}(x_{t+1}\mid x_{\le t})$
    - 在位置 $t$，主模型只预测下一个 token $x_{t+1}$
- MTP 会额外优化：$p(x_{t+2}\mid x_{\le t}),\ p(x_{t+3}\mid x_{\le t}),\ldots$
- 通常写成：$L = L_{\text{next}} + \lambda \frac{1}{D}\sum_{k=1}^{D} L_{\text{future},k}$
    - $\mathcal{L}_{\text{MTP}}^{k}=-\frac{1}{T}\sum_{i=2+k}^{T+1}\log P_i^k[t_i]$
        - $h_1$, MTP-1 额外看到的token是$t_2$，MTP-1 预测目标是 $t_3$
        - $h_2$, MTP-2 额外看到的token是$t_3$，MTP-2 预测目标是 $t_4$
        - $h_3$, MTP-3 额外看到的token是$t_4$，MTP-3 预测目标是 $t_5$
    - MTP 额外加 $D$个“未来深度”的预测头或预测模块。若把第 $k$ 个 MTP 深度记为 $L_{\text{future},k}$，它预测的是更远的 token：
    - $D$ 额外预测的未来 token 深度数量。
    - $D=1$，主模型 $x_t \rightarrow x_{t+1}$，还额外预测一个更远 token：$(x_t,x_{t+1}) \rightarrow x_{t+2}$
    - $D=2$，主模型预测 $x_{t+1}$，两个 MTP 分别预测：$x_{t+2},\quad x_{t+3}$
    - $L=L_{\text{next}}+\frac{\lambda}{2}\left(L_{\text{future},1}+L_{\text{future},2}\right)$
- 含义
    - 每个位置不只给模型一个监督信号，而是给多个未来 token 的监督信号。
    - MTP 一方面能“densify training signals”，另一方面能迫使表示提前为未来 token 做规划。
    - 同样 token budget 下模型学得更充分，或者达到同等能力可能需要更少训练 token。
    - v3 默认配置 $D=1$，multi-token prediction depth 设置为 1，也就是除了精确的 next token，每个 token 再预测一个额外 token。
$$
D=1,\quad L_{\text{total}}=L_{\text{next}}+\lambda L_{\text{MTP}}^{1},\quad\lambda=\begin{cases}
0.3, & \text{前 }10T\text{ tokens}\\
0.1, & \text{后 }4.8T\text{ tokens}
\end{cases}
$$
- sequential MTP module，保留 causal chain（auto regression 的过程）。也就是第 $k$ 个深度不是从同一个 hidden state 并行乱猜远处 token，而是把前一深度表示和中间 token embedding 结合后继续往后推。论文称其区别于并行 independent output heads，并强调它 sequentially predicts additional tokens。
- DeepSeek-V3 论文：MTP 主要目标是提升主模型性能，推理时可以直接丢弃 MTP 模块；也可以把 MTP 模块改作 speculative decoding 来降低生成延迟。见论文 MTP in Inference 段落：main model 可以独立正常工作，MTP modules 可以用于 speculative decoding 加速。

```
# single
主干 LLM:
h^0_t -> lm_head -> predict x_{t+1}

MTP depth 1:
h^0_t + embedding(x_{t+1}) -> MTP block 1 -> h^1_t -> head -> predict x_{t+2}

MTP depth 2:
h^1_t + embedding(x_{t+2}) -> MTP block 2 -> h^2_t -> head -> predict x_{t+3}

# batch
Main:
h^0_{1:4} = MainModel(E(t_{1:4}))
head(h^0_{1:4}) -> target t_{2:5}

MTP 1:
h^1_{1:4} = MTP1(h^0_{1:4}, E(t_{2:5}))
head(h^1_{1:4}) -> target t_{3:6}

MTP 2:
h^2_{1:4} = MTP2(h^1_{1:4}, E(t_{3:6}))
head(h^2_{1:4}) -> target t_{4:7}

# single of batch
Main position 1:
t1 -> main hidden h^0_1 -> predict t2

MTP depth 1, position 1:
h^0_1 + embedding(t2) -> h^1_1 -> predict t3

MTP depth 2, position 1:
h^1_1 + embedding(t3) -> h^2_1 -> predict t4
```

#### mtp vs. 探测解码

> DeepSeek-V3 的 MTP 用作 speculative decoding 时，可以看成一种“内置在 target model 里的 draft model”（self-speculative decoding），而 MTP blocks 的 causal chain 本质上就是一个短程自回归 draft 过程。(参考 mimo v2 flash)

- 经典 speculative decoding 是：
```
prefix c
  -> draft model q  自回归生成若干候选 token
  -> target model p 一次性验证这些候选 token
  -> 接受一个前缀，不接受处从 target 分布重采样
```

$$
\begin{split}
\hat{x}_{t+1}& \sim q(\cdot \mid c)\\
\hat{x}_{t+2}& \sim q(\cdot \mid c,\hat{x}_{t+1})\\
\hat{x}_{t+3}& \sim q(\cdot \mid c,\hat{x}_{t+1},\hat{x}_{t+2})
\end{split}
$$

- DeepSeek-V3 的 MTP 版本可以抽象成：
```
prefix c = x_1,...,x_t

main model:
h^0_t = F_0(c)
p_1 = softmax(O(h^0_t))
sample / choose xhat_{t+1}

MTP block 1:
h^1_t = M_1(h^0_t, E(xhat_{t+1}))
q_2 = softmax(O(h^1_t))
sample / choose xhat_{t+2}

MTP block 2:
h^2_t = M_2(h^1_t, E(xhat_{t+2}))
q_3 = softmax(O(h^2_t))
sample / choose xhat_{t+3}
```

#### vllm mtp

```sh
vllm serve XiaomiMiMo/MiMo-7B-Base \
  --tensor-parallel-size 1 \
  --speculative-config '{"method":"mtp","num_speculative_tokens":1}'
```

#### verl mtp

- https://verl.readthedocs.io/en/latest/advance/mtp.html
    - RL training can be performed on mimo-7B-RL, Qwen-next, and Deepseek series models based on the MTP architecture.
    - examples/sft/gsm8k/run_mimo_7b_mtp_megatron.sh
- mtp layer names
    - num_nextn_predict_layers: DeepSeek, Qwen3 style
    - mtp_num_hidden_layers: Qwen3.5 style
- train or rollout
    - enable_train=True，加 MTP auxiliary loss，让 draft/MTP 模块学会预测未来 token。
    - enable_rollout=True，把 MTP 模块作为 speculative decoding 的 draft path。
        - async_sglang_server.py / vllm_async_server.py 